# 03 — RAG Experiments

## Objective

This notebook investigates how prompt design and retrieval settings affect answer generation in a Simple RAG pipeline.

It builds on the implementation in `02_simple_rag.ipynb` and compares alternative prompting strategies while keeping the underlying corpus, embedding model, FAISS index, and language model unchanged.

## Experiments

1. Compare the baseline RAG prompt with a concise answer prompt.
2. Evaluate an evidence-focused prompt.
3. Evaluate a precise answer prompt.
4. Inspect retrieved evidence for selected questions.
5. Compare retrieval settings using `top_k=5` and `top_k=10`.

## Experimental Setup

- **Dataset:** HotpotQA
- **Evaluation set:** 10 questions
- **Embedding model:** `sentence-transformers/all-MiniLM-L6-v2`
- **Vector index:** FAISS `IndexFlatIP`
- **Language model:** Qwen2.5-7B-Instruct (8-bit quantization)

## Notes

These experiments are intended for learning and error analysis. Results are based on a small evaluation set and should not be interpreted as definitive evidence of general performance improvements.

The baseline implementation is documented in `02_simple_rag.ipynb`.

## Project Configuration


In [1]:
# Install the required packages
%pip install -q -U faiss-cpu sentence-transformers bitsandbytes accelerate transformers

In [2]:
# ============================================================
# Cell 1 — Project Configuration
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Basic_RAG")

DATASET_NAME = "hotpotqa"
TOP_K = 5
N_EVAL_SAMPLES = 10

DATA_DIR = PROJECT_ROOT / "data" / DATASET_NAME
CORPUS_DIR = PROJECT_ROOT / "corpus" / DATASET_NAME
INDEX_DIR = PROJECT_ROOT / "indexes" / DATASET_NAME
RESULTS_DIR = PROJECT_ROOT / "results" / DATASET_NAME

EVALUATION_FILE = DATA_DIR / "evaluation_10.json"
CORPUS_FILE = CORPUS_DIR / "corpus.json"
FAISS_INDEX_FILE = INDEX_DIR / "faiss.index"

print("Project configuration loaded.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_NAME}")
print(f"Top-k: {TOP_K}")
print(f"Evaluation samples: {N_EVAL_SAMPLES}")

Project configuration loaded.
Project root: /content/drive/MyDrive/Basic_RAG
Dataset: hotpotqa
Top-k: 5
Evaluation samples: 10


## Connect Google Drive and Check Required Files


In [3]:
# ============================================================
# Cell 2 — Connect Google Drive and Check Required Files
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

required_files = [
    EVALUATION_FILE,
    CORPUS_FILE,
    FAISS_INDEX_FILE,
]

missing_files = []

print("Checking required files...")

for file_path in required_files:
    if file_path.exists():
        print(f"[OK] {file_path}")
    else:
        print(f"[MISSING] {file_path}")
        missing_files.append(str(file_path))

if missing_files:
    raise FileNotFoundError(
        "Required files are missing:\n"
        + "\n".join(missing_files)
    )

print("All required files are available.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking required files...
[OK] /content/drive/MyDrive/Basic_RAG/data/hotpotqa/evaluation_10.json
[OK] /content/drive/MyDrive/Basic_RAG/corpus/hotpotqa/corpus.json
[OK] /content/drive/MyDrive/Basic_RAG/indexes/hotpotqa/faiss.index
All required files are available.


## Load the Existing Corpus and FAISS Index


In [4]:
# ============================================================
# Cell 3 — Load Existing Corpus and FAISS Index
# ============================================================

import json
import faiss
import numpy as np

# Load corpus
with open(CORPUS_FILE, "r", encoding="utf-8") as file:
    corpus = json.load(file)

# Load FAISS index
faiss_index = faiss.read_index(str(FAISS_INDEX_FILE))

# Load evaluation questions
with open(EVALUATION_FILE, "r", encoding="utf-8") as file:
    evaluation_data = json.load(file)

print("Existing data loaded successfully.")
print(f"Number of corpus chunks: {len(corpus)}")
print(f"Number of vectors in FAISS index: {faiss_index.ntotal}")
print(f"Embedding dimension: {faiss_index.d}")
print(f"Number of evaluation samples: {len(evaluation_data)}")

Existing data loaded successfully.
Number of corpus chunks: 995
Number of vectors in FAISS index: 995
Embedding dimension: 384
Number of evaluation samples: 10


## Prepare the Retriever


The `FAISSRetriever` class below is copied from Notebook 1 (`01_build_corpus_and_index.ipynb`). It is reused here to retrieve relevant text chunks from the existing FAISS index without rebuilding the corpus or generating new document embeddings.

This duplication is intentional, to keep this notebook self-contained and easy to run independently. The class can be moved to a shared Python module in a future refactoring step.

In [5]:
# ============================================================
# Cell 4 — Load Embedding Model and Initialize Retriever
# ============================================================

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

class FAISSRetriever:
    """
    Simple reusable retriever based on:
    SentenceTransformer + FAISS IndexFlatIP
    """

    def __init__(self, embedding_model, index, corpus):
        self.embedding_model = embedding_model
        self.index = index
        self.corpus = corpus

    def retrieve(self, query, top_k=5):
        """
        Retrieve the top-k most similar chunks for a query.
        """

        # Encode query
        query_embedding = self.embedding_model.encode(
            [query],
            normalize_embeddings=True
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype="float32"
        )

        # Search FAISS
        scores, indices = self.index.search(
            query_embedding,
            top_k
        )

        # Build readable results
        results = []

        for rank, (score, idx) in enumerate(
            zip(scores[0], indices[0]),
            start=1
        ):
            chunk = self.corpus[idx]

            result = {
                "rank": rank,
                "score": float(score),
                "chunk_id": chunk["chunk_id"],
                "sample_id": chunk["sample_id"],
                "title": chunk["title"],
                "text": chunk["text"],
                "dataset": chunk["dataset"],
                "source_type": chunk["source_type"]
            }

            results.append(result)

        return results

retriever = FAISSRetriever(
    embedding_model=embedding_model,
    index=faiss_index,
    corpus=corpus
)

print("Retriever initialized successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Retriever initialized successfully.


## Test Retrieval Before Using the LLM

Run retrieval for one question and inspect the returned passages, IDs, titles, and scores.

This is a retrieval-only smoke test; it should not call the language model.

In [6]:
# ============================================================
# Cell 5 — Test Retrieval Before Using LLM
# ============================================================

test_sample = evaluation_data[0]
query = test_sample["question"]

retrieved_chunks = retriever.retrieve(
    query=query,
    top_k=TOP_K
)

print(f"Question: {query}")
print(f"\nNumber of retrieved chunks: {len(retrieved_chunks)}")

for result in retrieved_chunks:
    print("\n" + "-" * 60)
    print(f"Rank: {result['rank']}")
    print(f"Score: {result['score']:.4f}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Title: {result['title']}")
    print(f"Text: {result['text'][:500]}")

Question: What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Number of retrieved chunks: 5

------------------------------------------------------------
Rank: 1
Score: 0.5720
Chunk ID: 5a7a567255429941d65f25bd_chunk_9
Title: Iqbal F. Qadir
Text: Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India.

------------------------------------------------------------
Rank: 2
Score: 0.5118
Chunk ID: 5a7a567255429941d65f25bd_chunk_4
Title: Mukachevo Radar Station
Text: Mukachevo radar station was a Soviet radar station providing early warning of ballistic missile attack.  It was located in Shipka in the far south west of Ukraine and was part of the Soviet, and then Russian 

## Build the RAG Prompt

Combine the user's question with the retrieved passages. The prompt should instruct the model to answer using the supplied evidence and to say when the evidence is insufficient.


In [7]:
# ============================================================
# Cell 6 — Build RAG Prompt
# ============================================================

def build_rag_prompt(query, retrieved_chunks):
    context_parts = []

    for result in retrieved_chunks:
        context_parts.append(
            f"[Source: {result['title']} | "
            f"Chunk ID: {result['chunk_id']}]\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""You are a helpful question-answering assistant.
Answer the question using only the provided context.
If the context does not contain enough information to answer,
clearly state that the available information is insufficient.
Do not invent facts.

Context:
{context}

Question:
{query}

Answer:"""

    return prompt


rag_prompt = build_rag_prompt(
    query=query,
    retrieved_chunks=retrieved_chunks
)

print("RAG prompt created successfully.")
print(f"Prompt length: {len(rag_prompt)} characters")
print("\nPrompt preview:\n")
print(rag_prompt[:2000])

RAG prompt created successfully.
Prompt length: 3088 characters

Prompt preview:

You are a helpful question-answering assistant.
Answer the question using only the provided context.
If the context does not contain enough information to answer,
clearly state that the available information is insufficient.
Do not invent facts.

Context:
[Source: Iqbal F. Qadir | Chunk ID: 5a7a567255429941d65f25bd_chunk_9]
Vice-Admiral Iqbal Fazl Quadir (Urdu:اقبال فضل قادر) , is a retired three-star rank admiral in the Pakistan Navy, former diplomat, and a defence analyst.  He is renown for his participation in second war with India when he was part of the flotilla that attacked the radar station in Dwarka, India.

[Source: Mukachevo Radar Station | Chunk ID: 5a7a567255429941d65f25bd_chunk_4]
Mukachevo radar station was a Soviet radar station providing early warning of ballistic missile attack.  It was located in Shipka in the far south west of Ukraine and was part of the Soviet, and then Russian missil

## Connect the Existing LLM Interface

Import `call_llm(prompt)` from `qwen_model.py`.


In [8]:
# ============================================================
# Cell 7 — Connect Existing LLM Interface
# ============================================================

import sys

MODEL_MODULE_DIR = Path("/content/drive/MyDrive")

if str(MODEL_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_MODULE_DIR))

from qwen_model import call_llm

print("LLM interface imported successfully.")

Loading Qwen tokenizer...
Model path: /content/drive/MyDrive/Model_qwen2.5-7B-instruct/models--Qwen--Qwen2.5-7B-Instruct/snapshots/a09a35458c702b33eeacc393d103063234e8bc28


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading Qwen2.5-7B-Instruct in 8-bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen model loaded successfully.
Device: cuda:0
VRAM allocated: 8.81 GB
LLM interface imported successfully.


## Run One End-to-End Simple RAG Query

We will start with a single question.

In [9]:
# ============================================================
# Cell 8 — Run One End-to-End Simple RAG Query
# ============================================================

answer = call_llm(rag_prompt)

print("Question:")
print(query)

print("\nGenerated Answer:")
print(answer)

print("\nRetrieved Sources:")

for result in retrieved_chunks:
    print(
        f"- {result['title']} "
        f"(Chunk ID: {result['chunk_id']}, "
        f"Score: {result['score']:.4f})"
    )

Question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Generated Answer:
The available information does not specify the location of the radar station attacked by Vice-Admiral Iqbal Fazl Quadir during the second war with India. The context mentions his participation in an attack on a radar station in Dwarka, India, but it does not provide details about its exact location on the Okhamandal Peninsula or confirm if it is on the western shore. Therefore, based on the given information, it is not possible to answer the question accurately.

Retrieved Sources:
- Iqbal F. Qadir (Chunk ID: 5a7a567255429941d65f25bd_chunk_9, Score: 0.5720)
- Mukachevo Radar Station (Chunk ID: 5a7a567255429941d65f25bd_chunk_4, Score: 0.5118)
- Sevastopol Radar Station (Chunk ID: 5a7a567255429941d65f25bd_chunk_8, Score: 0.4814)
- Radar Station B-71 (Chunk ID: 5a7a567255429941d65f25bd_chunk_6, Score: 0.4713)
- No. 227 Radar Sta

## Load the previously saved baseline results


In [10]:
# ============================================================
# Cell 9 — Load the previously saved baseline results
# ============================================================

import json
from pathlib import Path

RESULTS_DIR = Path(PROJECT_ROOT) / "results" / DATASET_NAME
BASELINE_RESULTS_FILE = RESULTS_DIR / "simple_rag_results.json"

if not BASELINE_RESULTS_FILE.exists():
    raise FileNotFoundError(
        f"Baseline results not found: {BASELINE_RESULTS_FILE}\n"
        "Run Notebook 02 first to generate the baseline results."
    )

with open(BASELINE_RESULTS_FILE, "r", encoding="utf-8") as file:
    evaluation_results = json.load(file)

print("Baseline results loaded successfully.")
print(f"Number of questions: {len(evaluation_results)}")
print(f"Results file: {BASELINE_RESULTS_FILE}")



Baseline results loaded successfully.
Number of questions: 10
Results file: /content/drive/MyDrive/Basic_RAG/results/hotpotqa/simple_rag_results.json


## Save Evaluation Results

In [ ]:
# ============================================================
# Cell 10 — Save Evaluation Results
# ============================================================

RESULTS_DIR = (
    Path(PROJECT_ROOT)
    / "results"
    / DATASET_NAME
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_FILE = RESULTS_DIR / "simple_rag_results.json"

with open(RESULTS_FILE, "w", encoding="utf-8") as file:
    json.dump(
        evaluation_results,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Evaluation results saved successfully.")
print(f"File path: {RESULTS_FILE}")
print(f"Number of saved results: {len(evaluation_results)}")


Evaluation results saved successfully.
File path: /content/drive/MyDrive/Basic_RAG/results/hotpotqa/simple_rag_results.json
Number of saved results: 10


## Review the Results


In [11]:
# ============================================================
# Cell 11 — Inspect Generated and Reference Answers
# ============================================================

for i, result in enumerate(evaluation_results, start=1):
    print(f"Question {i}")
    print("-" * 80)

    print("Question:")
    print(result["question"])

    print("\nReference Answer:")
    print(result["reference_answer"])

    print("\nGenerated Answer:")
    print(result["generated_answer"])

    print("\nGeneration Time:")
    print(f"{result['generation_time_seconds']:.2f} seconds")

    print("\n" + "=" * 80 + "\n")


Question 1
--------------------------------------------------------------------------------
Question:
What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?

Reference Answer:
flotilla

Generated Answer:
The available information does not specify the location of the radar station attacked by Vice-Admiral Iqbal Fazl Quadir during the second war with India. The context mentions his participation in an attack on a radar station in Dwarka, India, but it does not provide the exact location on the western shore of the Okhamandal Peninsula. Therefore, the available information is insufficient to answer the question.

Generation Time:
15.43 seconds


Question 2
--------------------------------------------------------------------------------
Question:
When did the park at which Tivolis Koncertsal is located open?

Reference Answer:
15 August 1843

Generated Answer:
Tivoli Gardens, the park at which Tivolis Koncertsal

In [12]:
# ============================================================
# Cell 12 — Calculate Exact Match and Token-level F1
# ============================================================

import re
import string

def normalize_answer(text):
    """Normalize an answer using common HotpotQA conventions."""
    text = str(text).lower()
    text = "".join(
        char for char in text
        if char not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = " ".join(text.split())
    return text


def calculate_exact_match(prediction, reference):
    return int(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def calculate_token_f1(prediction, reference):
    prediction_tokens = normalize_answer(prediction).split()
    reference_tokens = normalize_answer(reference).split()

    if not prediction_tokens or not reference_tokens:
        return float(prediction_tokens == reference_tokens)

    common_tokens = set(prediction_tokens) & set(reference_tokens)

    # Count repeated tokens correctly
    from collections import Counter
    common_count = sum(
        (Counter(prediction_tokens) & Counter(reference_tokens)).values()
    )

    if common_count == 0:
        return 0.0

    precision = common_count / len(prediction_tokens)
    recall = common_count / len(reference_tokens)

    return 2 * precision * recall / (precision + recall)


# Calculate metrics for each question
for result in evaluation_results:
    result["exact_match"] = calculate_exact_match(
        result["generated_answer"],
        result["reference_answer"]
    )
    result["token_f1"] = calculate_token_f1(
        result["generated_answer"],
        result["reference_answer"]
    )

# Aggregate results
num_questions = len(evaluation_results)

mean_em = (
    sum(result["exact_match"] for result in evaluation_results)
    / num_questions
)

mean_f1 = (
    sum(result["token_f1"] for result in evaluation_results)
    / num_questions
)

print(f"Number of evaluated questions: {num_questions}")
print(f"Exact Match: {mean_em:.4f} ({mean_em * 100:.2f}%)")
print(f"Mean Token-level F1: {mean_f1:.4f} ({mean_f1 * 100:.2f}%)")

print("\nPer-question metrics:")
for i, result in enumerate(evaluation_results, start=1):
    print(
        f"Q{i}: EM={result['exact_match']}, "
        f"F1={result['token_f1']:.4f}"
    )


Number of evaluated questions: 10
Exact Match: 0.0000 (0.00%)
Mean Token-level F1: 0.1231 (12.31%)

Per-question metrics:
Q1: EM=0, F1=0.0000
Q2: EM=0, F1=0.3529
Q3: EM=0, F1=0.0645
Q4: EM=0, F1=0.0000
Q5: EM=0, F1=0.0833
Q6: EM=0, F1=0.0000
Q7: EM=0, F1=0.2222
Q8: EM=0, F1=0.2000
Q9: EM=0, F1=0.3077
Q10: EM=0, F1=0.0000


In [15]:
# ============================================================
# Cell 13 — Summarize Simple RAG Evaluation
# ============================================================

num_questions = len(evaluation_results)

total_generation_time = sum(
    result["generation_time_seconds"]
    for result in evaluation_results
)

mean_generation_time = (
    total_generation_time / num_questions
    if num_questions else 0
)

answer_coverage = sum(
    result["reference_answer_in_context"]
    for result in evaluation_results
)

exact_match_count = sum(
    result["exact_match"]
    for result in evaluation_results
)

mean_f1 = (
    sum(result["token_f1"] for result in evaluation_results)
    / num_questions
    if num_questions else 0
)

print("Simple RAG Evaluation Summary")
print("=" * 50)
print(f"Number of questions: {num_questions}")
print(f"Reference answers found in retrieved context: "
      f"{answer_coverage}/{num_questions}")
print(f"Reference answer coverage: "
      f"{answer_coverage / num_questions * 100:.2f}%")
print(f"Exact Match: {exact_match_count / num_questions * 100:.2f}%")
print(f"Mean Token-level F1: {mean_f1 * 100:.2f}%")
print(f"Total generation time: {total_generation_time:.2f} seconds")
print(f"Average generation time: {mean_generation_time:.2f} seconds")


Simple RAG Evaluation Summary
Number of questions: 10
Reference answers found in retrieved context: 9/10
Reference answer coverage: 90.00%
Exact Match: 0.00%
Mean Token-level F1: 12.31%
Total generation time: 79.25 seconds
Average generation time: 7.92 seconds


## Experiment 1 — Prompt Design

We first compare different prompt designs while reusing the same retrieved chunks. This helps isolate the effect of prompt instructions from changes in retrieval.

In [16]:
# ============================================================
# Cell 14 — Build Concise Answer Prompt
# ============================================================

def build_concise_rag_prompt(query, retrieved_chunks):
    context_parts = []

    for result in retrieved_chunks:
        context_parts.append(
            f"[Source: {result['title']}]\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""You are a question-answering system.

Answer the question using only the information in the provided context.

Instructions:
- Give the shortest answer that fully answers the question.
- Prefer a name, date, place, number, or short phrase when appropriate.
- Do not include explanations, introductions, or concluding sentences.
- Do not repeat the question.
- Do not add facts that are not supported by the context.
- If the context does not provide enough evidence to answer,
  respond with exactly: Insufficient information.

Context:
{context}

Question:
{query}

Short answer:"""

    return prompt

print("Concise RAG prompt function created successfully.")


Concise RAG prompt function created successfully.


### Experiment Management and Result Caching

The prompt experiments can require multiple calls to the language model, which increases execution time. To avoid unnecessary repeated calls, this section introduces a shared function for managing experiment results.

For each experiment, the function checks whether previously saved results are available and match the current evaluation questions. If valid results exist, they are loaded from disk. Otherwise, the experiment is executed, and its results are saved for future use.

This mechanism allows the experiments to be reproduced when needed while making subsequent analysis faster. It also keeps result management consistent across the Concise, Evidence-Focused, and Precise Answer prompting methods.

**Note:** Cached results are reused only when the evaluation questions match. If the prompt, model configuration, or other experimental settings change, the corresponding cached results should be regenerated to ensure a fair comparison.

In [17]:
# ============================================================
# Cell 15 — Experiment Management and Result Caching
# ============================================================

import json
import time

def load_or_run_prompt_experiment(method_name, prompt_builder):
    """
    Load cached experiment results if available.
    Otherwise, run the experiment and save the results.
    """

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    results_file = RESULTS_DIR / f"{method_name}_results.json"

    # Load cached results if available
    if results_file.exists():
        with open(results_file, "r", encoding="utf-8") as file:
            saved_data = json.load(file)

        cached_results = (
            saved_data["results"]
            if isinstance(saved_data, dict) and "results" in saved_data
            else saved_data
        )

        if (
            isinstance(cached_results, list)
            and len(cached_results) == len(evaluation_results)
            and all(
                cached.get("question_id") == original.get("question_id")
                and cached.get("question") == original.get("question")
                for cached, original in zip(
                    cached_results, evaluation_results
                )
            )
        ):
            print(f"Loaded cached results: {method_name}")
            print(f"Number of questions: {len(cached_results)}")
            return cached_results

        print("Cached results do not match the current evaluation set.")
        print("Running the experiment again.")

    # Run the experiment if no valid cache exists
    experiment_results = []

    for i, original in enumerate(evaluation_results, start=1):
        retrieved_chunks = original["retrieved_chunks"]

        prompt = prompt_builder(
            query=original["question"],
            retrieved_chunks=retrieved_chunks
        )

        start_time = time.time()
        answer = call_llm(prompt)
        elapsed_time = time.time() - start_time

        experiment_results.append({
            "question_id": original["question_id"],
            "question": original["question"],
            "reference_answer": original["reference_answer"],
            "generated_answer": answer,
            "generation_time_seconds": round(elapsed_time, 2),
            "retrieved_chunks": retrieved_chunks
        })

        print(f"Completed {i}/{len(evaluation_results)}")
        print(f"Generation time: {elapsed_time:.2f} seconds")

    # Save results so the experiment does not need to run again
    with open(results_file, "w", encoding="utf-8") as file:
        json.dump(
            {
                "method": method_name,
                "dataset": DATASET_NAME,
                "top_k": TOP_K,
                "num_questions": len(experiment_results),
                "results": experiment_results
            },
            file,
            ensure_ascii=False,
            indent=2
        )

    print(f"Experiment completed: {method_name}")
    print(f"Results saved to: {results_file}")

    return experiment_results


**Evaluate Concise RAG**

In [18]:
# ============================================================
# Cell 16 — Evaluate Concise RAG
# ============================================================


concise_results = load_or_run_prompt_experiment(
    method_name="concise_rag",
    prompt_builder=build_concise_rag_prompt
)

print("\nConcise RAG evaluation completed.")
print(f"Total evaluated questions: {len(concise_results)}")


Loaded cached results: concise_rag
Number of questions: 10

Concise RAG evaluation completed.
Total evaluated questions: 10


## Compare Baseline and Concise RAG

In [19]:
# ============================================================
# Cell 17 — Compare Baseline and Concise RAG
# ============================================================

import re
import string
from collections import Counter

def normalize_answer(text):
    """Normalize an answer for Exact Match and token F1."""
    text = text.lower()
    text = "".join(
        char for char in text
        if char not in string.punctuation
    )
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match_score(prediction, reference):
    return int(
        normalize_answer(prediction)
        == normalize_answer(reference)
    )


def token_f1_score(prediction, reference):
    prediction_tokens = normalize_answer(prediction).split()
    reference_tokens = normalize_answer(reference).split()

    if not prediction_tokens or not reference_tokens:
        return int(prediction_tokens == reference_tokens)

    common = Counter(prediction_tokens) & Counter(reference_tokens)
    common_count = sum(common.values())

    if common_count == 0:
        return 0.0

    precision = common_count / len(prediction_tokens)
    recall = common_count / len(reference_tokens)

    return 2 * precision * recall / (precision + recall)


def evaluate_results(results):
    em_scores = []
    f1_scores = []

    for result in results:
        prediction = result["generated_answer"]
        reference = result["reference_answer"]

        em_scores.append(
            exact_match_score(prediction, reference)
        )
        f1_scores.append(
            token_f1_score(prediction, reference)
        )

    return {
        "EM": sum(em_scores) / len(em_scores),
        "Token F1": sum(f1_scores) / len(f1_scores),
        "EM scores": em_scores,
        "F1 scores": f1_scores,
    }


baseline_metrics = evaluate_results(evaluation_results)
concise_metrics = evaluate_results(concise_results)

print("Evaluation Comparison")
print("=" * 55)
print(f"{'Metric':<20}{'Baseline RAG':>15}{'Concise RAG':>15}")
print("-" * 55)
print(
    f"{'Exact Match (%)':<20}"
    f"{baseline_metrics['EM'] * 100:>14.2f}%"
    f"{concise_metrics['EM'] * 100:>14.2f}%"
)
print(
    f"{'Token F1 (%)':<20}"
    f"{baseline_metrics['Token F1'] * 100:>14.2f}%"
    f"{concise_metrics['Token F1'] * 100:>14.2f}%"
)
print("=" * 55)

print("\nPer-question comparison")
print("-" * 80)

for i, (baseline, concise) in enumerate(
    zip(evaluation_results, concise_results), start=1
):
    print(f"\nQuestion {i}")
    print(f"Reference: {concise['reference_answer']}")
    print(f"Baseline:  {baseline['generated_answer']}")
    print(f"Concise:   {concise['generated_answer']}")
    print(
        f"EM: {baseline_metrics['EM scores'][i-1]} -> "
        f"{concise_metrics['EM scores'][i-1]}"
    )
    print(
        f"F1: {baseline_metrics['F1 scores'][i-1]:.3f} -> "
        f"{concise_metrics['F1 scores'][i-1]:.3f}"
    )


Evaluation Comparison
Metric                 Baseline RAG    Concise RAG
-------------------------------------------------------
Exact Match (%)               0.00%         40.00%
Token F1 (%)                 12.31%         45.00%

Per-question comparison
--------------------------------------------------------------------------------

Question 1
Reference: flotilla
Baseline:  The available information does not specify the location of the radar station attacked by Vice-Admiral Iqbal Fazl Quadir during the second war with India. The context mentions his participation in an attack on a radar station in Dwarka, India, but it does not provide the exact location on the western shore of the Okhamandal Peninsula. Therefore, the available information is insufficient to answer the question.
Concise:   Insufficient information.
EM: 0 -> 0
F1: 0.000 -> 0.000

Question 2
Reference: 15 August 1843
Baseline:  Tivoli Gardens, the park at which Tivolis Koncertsal is located, opened on 15 August 1843.


## Build Evidence-Focused Prompt

In [20]:
# ============================================================
# Cell 18 — Build Evidence-Focused Prompt
# ============================================================

def build_evidence_focused_prompt(query, retrieved_chunks):
    context_parts = []

    for i, result in enumerate(retrieved_chunks, start=1):
        context_parts.append(
            f"[Evidence {i}: {result['title']}]\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""You are a precise question-answering system.

Answer the question using only the evidence in the context.

Follow these steps:
1. Identify the evidence that is directly relevant to the question.
2. Use that evidence to determine the answer.
3. Return only the shortest answer supported by the evidence.

Rules:
- Pay close attention to names, dates, titles, and specific descriptions.
- Do not substitute a related fact for the requested answer.
- Do not guess or use outside knowledge.
- If the evidence does not support a reliable answer, respond with exactly:
Insufficient information.
- Do not include reasoning or explanations in the final answer.

Context:
{context}

Question:
{query}

Answer:"""

    return prompt


print("Evidence-focused prompt function created successfully.")


Evidence-focused prompt function created successfully.


In [21]:
# ============================================================
# Cell 19 — Evaluate Evidence-Focused RAG
# ============================================================


evidence_focused_results = load_or_run_prompt_experiment(
    method_name="evidence_focused_rag",
    prompt_builder=build_evidence_focused_prompt
)


print("\nEvidence-focused RAG evaluation completed.")
print(f"Total evaluated questions: {len(evidence_focused_results)}")


Loaded cached results: evidence_focused_rag
Number of questions: 10

Evidence-focused RAG evaluation completed.
Total evaluated questions: 10


## Comparison: Baseline, Concise, and Evidence-Focused RAG

Compare Exact Match and token-level F1 for the three methods. Also check whether the two alternative prompts produce different answers.

In [22]:
# ============================================================
# Cell 20 — Compare All Three RAG Methods
# ============================================================

methods = {
    "Baseline RAG": evaluation_results,
    "Concise RAG": concise_results,
    "Evidence-Focused RAG": evidence_focused_results,
}

print("Three-Method Evaluation Comparison")
print("=" * 65)
print(f"{'Method':<25}{'Exact Match (%)':>20}{'Token F1 (%)':>18}")
print("-" * 65)

for method_name, results in methods.items():
    metrics = evaluate_results(results)

    print(
        f"{method_name:<25}"
        f"{metrics['EM'] * 100:>19.2f}%"
        f"{metrics['Token F1'] * 100:>17.2f}%"
    )

print("=" * 65)

# Check whether the concise and evidence-focused answers are identical
same_answers = all(
    normalize_answer(concise["generated_answer"])
    == normalize_answer(evidence["generated_answer"])
    for concise, evidence in zip(
        concise_results, evidence_focused_results
    )
)

print(
    "Concise and evidence-focused answers identical:",
    same_answers
)


Three-Method Evaluation Comparison
Method                        Exact Match (%)      Token F1 (%)
-----------------------------------------------------------------
Baseline RAG                            0.00%            12.31%
Concise RAG                            40.00%            45.00%
Evidence-Focused RAG                   40.00%            45.00%
Concise and evidence-focused answers identical: True


In [23]:
# ============================================================
# Cell 21 — Inspect Evidence for Questions 2 and 4
# ============================================================

questions_to_inspect = [2, 4]

for question_number in questions_to_inspect:
    result = evaluation_results[question_number - 1]

    print("=" * 80)
    print(f"QUESTION {question_number}")
    print("-" * 80)
    print(f"Question: {result['question']}")
    print(f"Reference answer: {result['reference_answer']}")
    print(f"Concise answer: {concise_results[question_number - 1]['generated_answer']}")
    print()

    for chunk in result["retrieved_chunks"]:
        print(
            f"Rank: {chunk['rank']} | "
            f"Title: {chunk['title']} | "
            f"Score: {chunk['score']:.4f}"
        )
        print(chunk["text"])
        print("-" * 80)


QUESTION 2
--------------------------------------------------------------------------------
Question: When did the park at which Tivolis Koncertsal is located open?
Reference answer: 15 August 1843
Concise answer: 1843

Rank: 1 | Title: Tivolis Koncertsal | Score: 0.7277
Tivolis Koncertsal is a 1,660-capacity concert hall located at Tivoli Gardens in Copenhagen, Denmark.  The building, which was designed by Frits Schlegel and Hans Hansen, was built between 1954 and 1956.
--------------------------------------------------------------------------------
Rank: 2 | Title: Tivoli Gardens | Score: 0.6321
Tivoli Gardens (or simply Tivoli) is a famous amusement park and pleasure garden in Copenhagen, Denmark.  The park opened on 15 August 1843 and is the second-oldest operating amusement park in the world, after Dyrehavsbakken in nearby Klampenborg, also in Denmark.
--------------------------------------------------------------------------------
Rank: 3 | Title: Tivoli Two | Score: 0.4541
Tivol

### Experiment 2 — Precise Answer Prompt

This experiment tests whether more explicit instructions about dates, titles, and directly relevant facts improve answer accuracy. The prompt may also become too restrictive, so both improvements and regressions should be examined.

In [24]:
# ============================================================
# Cell 22 — Build Precise Answer Prompt
# ============================================================

def build_precise_answer_prompt(query, retrieved_chunks):
    context_parts = []

    for result in retrieved_chunks:
        context_parts.append(
            f"[Source: {result['title']}]\n"
            f"{result['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""You are a precise question-answering system.

Answer the question using only the provided context.

Instructions:
- Identify exactly what information the question asks for.
- Select the statement that directly answers that question.
- Do not confuse different facts about the same person, place, or organization.
- Include all necessary details from the context.
- For a question asking when an event happened, return the full date
  if the day, month, and year are available.
- For a question asking for a designation or title, return the exact
  designation stated in the context.
- Return only the answer, without explanations.
- Do not guess or use outside knowledge.
- If the answer cannot be supported by the context, return exactly:
  Insufficient information.

Context:
{context}

Question:
{query}

Answer:"""

    return prompt


print("Precise answer prompt function created successfully.")


Precise answer prompt function created successfully.


In [25]:
# ============================================================
# Cell 23 — Test Precise Answer Prompt on Questions 2 and 4
# ============================================================

precise_test_results = []

for question_number in [2, 4]:
    result = evaluation_results[question_number - 1]

    prompt = build_precise_answer_prompt(
        query=result["question"],
        retrieved_chunks=result["retrieved_chunks"]
    )

    start_time = time.time()
    answer = call_llm(prompt)
    elapsed_time = time.time() - start_time

    test_result = {
        "question_id": result["question_id"],
        "question": result["question"],
        "reference_answer": result["reference_answer"],
        "generated_answer": answer,
        "generation_time_seconds": round(elapsed_time, 2),
        "retrieved_chunks": result["retrieved_chunks"]
    }

    precise_test_results.append(test_result)

    print(f"Question {question_number}")
    print(f"Reference: {result['reference_answer']}")
    print(f"Previous answer: {concise_results[question_number - 1]['generated_answer']}")
    print(f"New answer: {answer}")
    print(f"Generation time: {elapsed_time:.2f} seconds")
    print("-" * 60)

print("Precise answer prompt test completed.")


Question 2
Reference: 15 August 1843
Previous answer: 1843
New answer: 15 August 1843
Generation time: 1.96 seconds
------------------------------------------------------------
Question 4
Reference: Centre of Excellence
Previous answer: good
New answer: Centre of Excellence
Generation time: 1.18 seconds
------------------------------------------------------------
Precise answer prompt test completed.


In [26]:
# ============================================================
# Cell 24 — Evaluate Precise Answer RAG
# ============================================================


precise_results = load_or_run_prompt_experiment(
    method_name="precise_answer_rag",
    prompt_builder=build_precise_answer_prompt
)


print("\nPrecise Answer RAG evaluation completed.")
print(f"Total evaluated questions: {len(precise_results)}")


Loaded cached results: precise_answer_rag
Number of questions: 10

Precise Answer RAG evaluation completed.
Total evaluated questions: 10


In [27]:
# ============================================================
# Cell 25 — Compare All RAG Prompting Methods
# ============================================================

methods = {
    "Baseline RAG": evaluation_results,
    "Concise RAG": concise_results,
    "Evidence-Focused RAG": evidence_focused_results,
    "Precise Answer RAG": precise_results,
}

all_metrics = {}

print("RAG Method Comparison")
print("=" * 75)
print(f"{'Method':<25}{'Exact Match (%)':>20}{'Token F1 (%)':>18}")
print("-" * 75)

for method_name, results in methods.items():
    metrics = evaluate_results(results)
    all_metrics[method_name] = metrics

    print(
        f"{method_name:<25}"
        f"{metrics['EM'] * 100:>19.2f}%"
        f"{metrics['Token F1'] * 100:>17.2f}%"
    )

print("=" * 75)

# Compare answers question by question
print("\nPer-question comparison")
print("=" * 90)

for i in range(len(evaluation_results)):
    reference = evaluation_results[i]["reference_answer"]

    print(f"\nQuestion {i + 1}: {evaluation_results[i]['question']}")
    print(f"Reference: {reference}")

    for method_name, results in methods.items():
        answer = results[i]["generated_answer"]
        em = all_metrics[method_name]["EM scores"][i]

        print(
            f"{method_name}: {answer} "
            f"| EM: {em}"
        )


RAG Method Comparison
Method                        Exact Match (%)      Token F1 (%)
---------------------------------------------------------------------------
Baseline RAG                            0.00%            12.31%
Concise RAG                            40.00%            45.00%
Evidence-Focused RAG                   40.00%            45.00%
Precise Answer RAG                     40.00%            40.00%

Per-question comparison

Question 1: What was Iqbal F. Qadir on when he participated in an attack on a radar station located on western shore of the Okhamandal Peninsula?
Reference: flotilla
Baseline RAG: The available information does not specify the location of the radar station attacked by Vice-Admiral Iqbal Fazl Quadir during the second war with India. The context mentions his participation in an attack on a radar station in Dwarka, India, but it does not provide the exact location on the western shore of the Okhamandal Peninsula. Therefore, the available information is 

## Saving All Results

In [28]:
# ============================================================
# Cell 26 — Saving All Results
# ============================================================

import os
import json
import csv
from collections import Counter

# --------------------------------------------------
# 1. Configure output directory
# --------------------------------------------------
os.makedirs(RESULTS_DIR, exist_ok=True)

# --------------------------------------------------
# 2. Define evaluation methods
# --------------------------------------------------
results_by_method = {
    "baseline_rag": evaluation_results,
    "concise_rag": concise_results,
    "evidence_focused_rag": evidence_focused_results,
    "precise_answer_rag": precise_results,
}

# --------------------------------------------------
# 3. Define normalized EM and token-level F1
# --------------------------------------------------
def calculate_em_f1(prediction, reference):
    prediction = normalize_answer(str(prediction or ""))
    reference = normalize_answer(str(reference or ""))

    em = int(prediction == reference)

    pred_tokens = prediction.split()
    ref_tokens = reference.split()

    if not pred_tokens or not ref_tokens:
        return em, 0.0

    common = Counter(pred_tokens) & Counter(ref_tokens)
    common_count = sum(common.values())

    if common_count == 0:
        return em, 0.0

    precision = common_count / len(pred_tokens)
    recall = common_count / len(ref_tokens)
    f1 = 2 * precision * recall / (precision + recall)

    return em, f1

# --------------------------------------------------
# 4. Save each method and calculate metrics
# --------------------------------------------------
summary_rows = []
per_question_rows = []

for method_name, method_results in results_by_method.items():
    if not method_results:
        raise ValueError(f"No results found for {method_name}.")

    method_em = []
    method_f1 = []

    for item in method_results:
        prediction = item.get("generated_answer", "")
        reference = item.get("reference_answer", "")

        em, f1 = calculate_em_f1(prediction, reference)
        method_em.append(em)
        method_f1.append(f1)

        per_question_rows.append({
            "method": method_name,
            "question_id": item.get("question_id", ""),
            "question": item.get("question", ""),
            "reference_answer": reference,
            "generated_answer": prediction,
            "exact_match": em,
            "token_f1": round(f1, 4),
        })

    n = len(method_results)
    metrics = {
        "method": method_name,
        "num_questions": n,
        "exact_match": round(sum(method_em) / n, 4),
        "token_f1": round(sum(method_f1) / n, 4),
    }

    summary_rows.append(metrics)

    output_file = os.path.join(
        RESULTS_DIR, f"{method_name}_results.json"
    )

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            {
                "method": method_name,
                "dataset": DATASET_NAME,
                "top_k": TOP_K,
                "num_questions": n,
                "metrics": metrics,
                "results": method_results,
            },
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

# --------------------------------------------------
# 5. Save summary and per-question comparison
# --------------------------------------------------
metadata = {
    "dataset": DATASET_NAME,
    "top_k": TOP_K,
    "num_evaluation_samples": len(evaluation_data),
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "metric_note": (
        "EM and token-level F1 use normalized answers. "
        "This is a small 10-question diagnostic evaluation."
    ),
    "methods": summary_rows,
}

summary_file = os.path.join(RESULTS_DIR, "prompt_comparison_summary.json")
with open(summary_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

csv_file = os.path.join(RESULTS_DIR, "per_question_comparison.csv")
with open(csv_file, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=per_question_rows[0].keys())
    writer.writeheader()
    writer.writerows(per_question_rows)

# --------------------------------------------------
# 6. Display saved files and summary
# --------------------------------------------------
print("Results saved successfully.")
print(f"Output directory: {RESULTS_DIR}")
print("\nSaved files:")

for filename in sorted(os.listdir(RESULTS_DIR)):
    print(f"- {filename}")

print("\nMethod comparison:")
for row in summary_rows:
    print(
        f"{row['method']}: "
        f"EM={row['exact_match']:.1%}, "
        f"F1={row['token_f1']:.1%}, "
        f"N={row['num_questions']}"
    )

Results saved successfully.
Output directory: /content/drive/MyDrive/Basic_RAG/results/hotpotqa

Saved files:
- baseline_rag_results.json
- concise_rag_results.json
- evidence_focused_rag_results.json
- per_question_comparison.csv
- precise_answer_rag_results.json
- prompt_comparison_summary.json
- retrieval_baseline_top5.json
- simple_rag_results.json

Method comparison:
baseline_rag: EM=0.0%, F1=12.3%, N=10
concise_rag: EM=40.0%, F1=45.0%, N=10
evidence_focused_rag: EM=40.0%, F1=45.0%, N=10
precise_answer_rag: EM=40.0%, F1=40.0%, N=10


In [29]:
# ============================================================
# Cell 27 — Inspect the retrieval results for evaluation question 10
# ============================================================

question_index = 9
sample = evaluation_data[question_index]

question = sample["question"]
reference_answer = sample["answer"]

retrieved_chunks = retriever.retrieve(question, top_k=5)

print("Question:")
print(question)

print("\nReference answer:")
print(reference_answer)

print("\nRetrieved chunks:")
for chunk in retrieved_chunks:
    print("\n" + "=" * 80)
    print(f"Rank: {chunk['rank']}")
    print(f"Score: {chunk['score']:.4f}")
    print(f"Title: {chunk['title']}")
    print(f"Chunk ID: {chunk['chunk_id']}")
    print("Text:")
    print(chunk["text"])

Question:
"Tunak", is a bhangra/pop love song by an artist born in which year ?

Reference answer:
1967

Retrieved chunks:

Rank: 1
Score: 0.7476
Title: Tunak Tunak Tun
Chunk ID: 5ab82d095542990e739ec853_chunk_1
Text:
"Tunak Tunak Tun" (Punjabi: ਤੁਣਕ ਤੁਣਕ ਤੁਣ ) or simply "Tunak", is a bhangra/pop love song by Indian artist Daler Mehndi released in 1998.  At the time, critics complained that Mehndi's music was only popular due to his videos that featured beautiful women dancing.  Mehndi's response was to create a video that featured only himself.  The music video was the first made in India using greenscreen technology, which allowed the singer to superimpose his image over various computer-generated backgrounds such as desert and mountain landscapes and St. Basil's Cathedral.

Rank: 2
Score: 0.4211
Title: Lum's Love Song
Chunk ID: 5ab82d095542990e739ec853_chunk_7
Text:
Lum no Love Song (ラムのラブソング / "Love Song of Lum") is the debut single of Japanese pop singer Yuko Matsutani.  The singl

## Experiment 3 — Retrieval Depth

This experiment investigates whether retrieving more chunks provides evidence that is missing from the top five results.

We focus on Question 10 as a diagnostic case. This is a targeted investigation, not a full evaluation of both retrieval settings across the entire dataset.

In [30]:
# ============================================================
# Cell 27 — Compare retrieval with top_k=5 and top_k=10 for question 10
# ============================================================

question_index = 9
question = evaluation_data[question_index]["question"]
reference_answer = evaluation_data[question_index]["answer"]

results_top5 = retriever.retrieve(question, top_k=5)
results_top10 = retriever.retrieve(question, top_k=10)

print("Question:")
print(question)

print("\nReference answer:")
print(reference_answer)

print("\nAdditional chunks retrieved at ranks 6-10:")

for chunk in results_top10[5:]:
    print("\n" + "-" * 80)
    print(f"Rank: {chunk['rank']}")
    print(f"Score: {chunk['score']:.4f}")
    print(f"Title: {chunk['title']}")
    print(f"Text: {chunk['text']}")

print("\nCheck whether the reference year appears in retrieved text:")
reference_year_found = any(
    reference_answer in chunk["text"]
    for chunk in results_top10
)

print(f"Reference year found in top 10: {reference_year_found}")

Question:
"Tunak", is a bhangra/pop love song by an artist born in which year ?

Reference answer:
1967

Additional chunks retrieved at ranks 6-10:

--------------------------------------------------------------------------------
Rank: 6
Score: 0.3536
Title: Cho Yong-pil
Text: Cho Yong-pil (hangul: 조용필; hanja: 趙容弼; also written Jo, Yong-pil) is a South Korean pop singer born in 1950.  Many Korean Pop fans believe that Cho is one of the most influential figures in Korean pop music and has produced many hits of Korean pop music history, including "Return to Busan Port", "Dear Friend" and "The Lady Outside the Window".

--------------------------------------------------------------------------------
Rank: 7
Score: 0.3506
Title: Crazy in Love
Text: "Crazy in Love" is a song by American singer Beyoncé featuring American rapper Jay-Z, from Beyoncé's debut solo studio album "Dangerously in Love" (2003).  Both artists wrote and composed the song in collaboration with Rich Harrison and Eugene R

In [31]:
# ============================================================
# Cell 27 — Compare Concise RAG with top_k=5 and top_k=10
# ============================================================

question_index = 9
sample = evaluation_data[question_index]

question = sample["question"]
reference_answer = sample["answer"]

comparison_results = {}

for k in [5, 10]:
    chunks = retriever.retrieve(question, top_k=k)
    prompt = build_concise_rag_prompt(question, chunks)

    print(f"\nGenerating answer with top_k={k}...")

    generated_answer = call_llm(prompt)

    em, f1 = calculate_em_f1(generated_answer, reference_answer)

    comparison_results[str(k)] = {
        "question": question,
        "reference_answer": reference_answer,
        "generated_answer": generated_answer,
        "top_k": k,
        "exact_match": em,
        "token_f1": round(f1, 4),
        "retrieved_chunk_ids": [
            chunk["chunk_id"] for chunk in chunks
        ],
    }

    print(f"Answer: {generated_answer}")
    print(f"Exact Match: {em}")
    print(f"Token F1: {f1:.4f}")

print("\nComparison completed.")


Generating answer with top_k=5...
Answer: Insufficient information
Exact Match: 0
Token F1: 0.0000

Generating answer with top_k=10...
Answer: 1967
Exact Match: 1
Token F1: 1.0000

Comparison completed.
